<a href="https://colab.research.google.com/github/WallofWonder/Hugging-Face-RL-Learning/blob/main/notebooks/bonus-unit1/bonus-unit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bonus Unit 1: 让我们训练小狗 Huggy 🐶 去捡木棍

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit2/thumbnail.png" alt="Bonus Unit 1Thumbnail">

在这个笔记本中，我们将通过**教小狗 Huggy 捡木棍并直接在浏览器中与它玩耍**，来巩固我们在第一单元中学到的知识。

⬇️ 这是你**在单元结束时将实现的效果。** ⬇️（点击 ▶ 观看）

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.mp4" type="video/mp4"></video>

### 游戏环境 🎮

- Huggy the Dog，由 [Thomas Simonini](https://twitter.com/ThomasSimonini) 基于 [Puppo The Corgi](https://blog.unity.com/technology/puppo-the-corgi-cuteness-overload-with-the-unity-ml-agents-toolkit) 开发。

### 使用的库 📚

- [MLAgents](https://github.com/Unity-Technologies/ml-agents)

我们一直在努力改进我们的教程，因此**如果你在这个笔记本中发现了一些问题**，请在 [Github Repo 上开启一个 Issue](https://github.com/huggingface/deep-rl-class/issues)。

## 本笔记本的目标 🏆

在学习完本笔记本后，你将：

- 理解**用于训练 Huggy 的状态空间、动作空间和奖励函数**。
- **训练你自己的 Huggy** 去捡木棍。
- 能够**直接在浏览器中与你训练好的 Huggy 玩耍**。

## 本笔记本来自深度强化学习课程
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这个免费课程中，你将：

- 📖 在**理论和实践**中学习深度强化学习。
- 🧑‍💻 学习**使用著名的深度强化学习库**，如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在**独特的环境中训练智能体**

更多内容请查看 📚 教学大纲 👉 https://simoninithomas.github.io/deep-rl-course

不要忘记 **<a href="http://eepurl.com/ic5ZUD">注册课程</a>**（我们会收集你的电子邮件，以便在每个单元发布时向你发送链接，并为你提供有关挑战和更新的信息）。

保持联系的最佳方式是加入我们的 Discord 服务器，与社区和我们交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 前提条件 🏗️

在开始本笔记本之前，你需要：

🔲 📚 **通过学习第一单元，理解强化学习的基础知识**（MC、TD、奖励假设...）

🔲 📚 **通过学习 Bonus Unit 1，阅读关于 Huggy 的介绍**

## 设置 GPU 💪
- 为了**加速智能体的训练，我们将使用 GPU**。为此，请转到 `Runtime > Change Runtime type` (运行时 > 更改运行时类型)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- 选择 `Hardware Accelerator > GPU` (硬件加速器 > GPU)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## 克隆仓库 🔽

- 我们需要克隆包含 **ML-Agents** 的仓库。

In [ ]:
%%capture
# Clone the repository (can take 3min)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

## 设置虚拟环境 🔽
- 为了使 **ML-Agents** 在 Colab 中成功运行，Colab 的 Python 版本必须符合该库的 Python 要求。

- 我们可以在 `setup.py` 文件的 `python_requires` 参数下查看支持的 Python 版本。设置 **ML-Agents** 库所需的这些文件位于以下位置：
  - `/content/ml-agents/ml-agents/setup.py`
  - `/content/ml-agents/ml-agents-envs/setup.py`

- Colab 当前的 Python 版本（可以使用 `!python --version` 查看）与库的 `python_requires` 参数不匹配，因此安装可能会静默失败，并导致稍后执行命令时出现如下错误：
  - `/bin/bash: line 1: mlagents-learn: command not found`
  - `/bin/bash: line 1: mlagents-push-to-hf: command not found`

- 为了解决这个问题，我们将创建一个 Python 版本与 **ML-Agents** 库兼容的虚拟环境。

`注意：` *为了将来的兼容性，请务必检查安装文件中的 `python_requires` 参数。如果 Colab 的 Python 版本不兼容，请在下方的脚本中将虚拟环境设置为支持的最高 Python 版本。*

In [ ]:
# Colab's Current Python Version (Incompatible with ML-Agents)
!python --version

In [ ]:
# Install virtualenv and create a virtual environment
!pip install virtualenv
!virtualenv myenv

# Download and install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

In [ ]:
# Activate Miniconda and install Python ver 3.10.12
!source /usr/local/bin/activate
# 同意通道使用条款
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
# 切换制定 Python 版本
!conda install -q -y --prefix /usr/local python=3.10.12 ujson  # Specify the version here

# Set environment variables for Python and conda paths
!export PYTHONPATH=/usr/local/lib/python3.10/site-packages/
!export CONDA_PREFIX=/usr/local/envs/myenv

In [ ]:
# Python Version in New Virtual Environment (Compatible with ML-Agents)
!python --version

## 安装依赖项 🔽

In [ ]:
%%capture
# Go inside the repository and install the package (can take 3min)
%cd ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

## 下载环境压缩包并移动至 `./trained-envs-executables/linux/`

- 我们的环境可执行文件存储在压缩包中。
- 我们需要下载它并将其放置在 `./trained-envs-executables/linux/` 目录下。

In [ ]:
!pwd

In [ ]:
!mkdir ./trained-envs-executables
!mkdir ./trained-envs-executables/linux

我们使用 `wget` 从 https://github.com/huggingface/Huggy 下载 `Huggy.zip` 文件。

In [ ]:
!wget "https://github.com/huggingface/Huggy/raw/main/Huggy.zip" -O ./trained-envs-executables/linux/Huggy.zip

In [ ]:
%%capture
!unzip -d ./trained-envs-executables/linux/ ./trained-envs-executables/linux/Huggy.zip

确保你的文件是可访问的。

In [ ]:
!chmod -R 755 ./trained-envs-executables/linux/Huggy

## 让我们回顾一下这个环境是如何运作的

### 状态空间：Huggy “感知”到了什么

Huggy 并不直接“看到”他的环境。相反，我们向他提供有关环境的信息：

- 目标（木棍）的位置
- 他自己与目标之间的相对位置
- 他腿部的朝向

基于所有这些信息，Huggy **可以决定下一步采取什么行动来实现他的目标**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.jpg" alt="Huggy" width="100%">

### 动作空间：Huggy 可以做出的动作
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-action.jpg" alt="Huggy action" width="100%">

**关节电机驱动着 Huggy 的腿**。这意味着为了获得目标，Huggy 需要**学习如何正确旋转每条腿的关节电机，以便他能够移动**。

### 奖励函数

奖励函数的设计旨在让 **Huggy 完成他的目标**：捡回木棍。

记住，强化学习的基础之一是*奖励假设*：目标可以描述为**期望累计奖励的最大化**。

在这里，我们的目标是让 Huggy **跑向木棍，但不要旋转得太厉害**。因此，我们的奖励函数必须体现这一目标。

我们的奖励函数：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/reward.jpg" alt="Huggy reward function" width="100%">

- *朝向奖励*：当他**靠近目标时，我们给予奖励**。
- *时间惩罚*：在每次动作时给予固定的时间惩罚，以**迫使他尽快拿到木棍**。
- *旋转惩罚*：如果 Huggy **旋转太厉害或转向太快**，我们会给予惩罚。
- *到达目标奖励*：当 Huggy **触及目标**时，我们给予奖励。

## 创建 Huggy 配置文件

- 在 ML-Agents 中，你在 **config.yaml 文件中定义训练超参数**。

- 在本笔记本的范围内，我们不会修改超参数，但如果你想进行实验，也可以尝试修改其他超参数。Unity 提供了非常[详细的文档来解释其中的每一个参数](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md)。

- 我们需要为 Huggy 创建一个配置文件。

  - 点击屏幕左侧的文件夹图标。

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create_file.png" alt="Create file" width="10%">

  - 前往 `/content/ml-agents/config/ppo` 目录。
  - 右键点击并创建一个名为 `Huggy.yaml` 的新文件。

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create-huggy.png" alt="Create huggy.yaml" width="20%">

- 复制并粘贴以下内容 🔽

In [ ]:
behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000

### 🔍 Huggy.yaml 配置深度解析

这份 YAML 文件定义了 Huggy 的训练“规则”。以下是关键参数的含义：

#### 1. **Behaviors (行为)**
*   **Huggy**: 这里的名称必须与 Unity 项目中 `Behavior Parameters` 组件的名称一致。
*   **trainer_type: ppo**: 使用近端策略优化算法（Proximal Policy Optimization），这是一种既稳定又高效的强化学习算法。

#### 2. **Hyperparameters (超参数)**
*   **batch_size (2048)**: 每次梯度下降更新时使用的样本数量。对于复杂运动（如狗的四肢协调），较大的 batch size 通常更稳定。
*   **buffer_size (20480)**: 在更新模型之前收集的经验总量。通常是 batch size 的倍数。
*   **learning_rate (0.0003)**: 学习率，决定了模型更新的步长。0.0003 是 PPO 的经典默认值。
*   **beta (0.005)**: 熵正则化系数。它鼓励智能体进行**探索**。如果这个值太小，Huggy 可能会过早陷入局部最优解（比如只会原地打转）。
*   **epsilon (0.2)**: PPO 裁剪范围，防止模型在更新时变动过大，保证训练稳定性。
*   **lambd (0.95)**: 用于广义优势估计（GAE），平衡偏差和方差。

#### 3. **Network Settings (网络设置)**
*   **normalize: true**: 自动对输入的状态数据进行归一化处理，这对于处理不同单位的输入（如坐标和旋转角度）非常有用。
*   **hidden_units (512)** & **num_layers (3)**: 神经网络的深度和宽度。Huggy 的动作涉及多个关节，需要较强的神经网络来模拟复杂的函数。

#### 4. **Reward Signals (奖励信号)**
*   **gamma (0.995)**: 折扣因子。它决定了智能体对“未来奖励”的重视程度。0.995 意味着它非常看重长远目标（拿到木棍）。

#### 5. **Training Limits (训练限制)**
*   **max_steps (2e6)**: 总训练步数（200万步）。达到这个数字后训练会自动停止。
*   **checkpoint_interval (200000)**: 每隔 20 万步保存一次模型快照（.onnx 文件）。
*   **time_horizon (1000)**: 在将经验添加到经验池之前，智能体在一个序列中收集多少步数据。

- 别忘了保存文件！

- **如果你想修改超参数**，在 Google Colab 笔记本中，你可以点击这里打开 config.yaml：`/content/ml-agents/config/ppo/Huggy.yaml`

- 例如，**如果你想在训练期间保存更多模型**（目前我们每 200,000 个训练步长保存一次）。你需要修改：
  - `checkpoint_interval`: 每个检查点之间收集的训练步数。
  - `keep_checkpoints`: 保留的模型检查点的最大数量。

=> 请记住，**减小 `checkpoint_interval` 意味着需要上传到 Hub 的模型更多，因此上传时间也会更长**。
我们现在准备好训练我们的智能体了 🔥。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 训练我们的智能体

要训练我们的智能体，我们只需要**启动 mlagents-learn 并选择包含环境的可执行文件**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mllearn.png" alt="ml learn function" width="100%">

使用 ML Agents，我们运行一个训练脚本。我们定义四个参数：

1. `mlagents-learn <config>`: 超参数配置文件的路径。
2. `--env`: 环境可执行文件的位置。
3. `--run-id`: 你想给这次训练运行起的名称 ID。
4. `--no-graphics`: 在训练期间不启动可视化界面。

训练模型并使用 `--resume` 标志在发生中断时继续训练。

> 当你使用 `--resume` 时，第一次运行可能会失败，尝试再次运行该代码块以跳过错误。

根据你的机器性能，训练将持续 30 到 45 分钟（别忘了**设置 GPU**），去喝杯咖啡 ☕️，这是你应得的 🤗。

In [ ]:
!mlagents-learn ../config/ppo/Huggy.yaml --env=./trained-envs-executables/linux/Huggy/Huggy --run-id="Huggy2" --no-graphics

## 将智能体推送到 🤗 Hub

- 现在我们已经训练好了智能体，我们**准备好将其推送到 Hub，以便能够在浏览器中与 Huggy 玩耍🔥。**

为了能够与社区分享你的模型，还有三个步骤需要遵循：

1️⃣ （如果尚未完成）在 HF 创建一个账户 ➡ https://huggingface.co/join

2️⃣ 登录，然后你需要存储来自 Hugging Face 网站的身份验证令牌。
- 创建一个新的令牌 (https://huggingface.co/settings/tokens)，**具有写（write）权限**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制令牌。
- 运行下面的单元格并粘贴令牌。

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

如果你不想使用 Google Colab 或 Jupyter Notebook，你需要改用此命令：`huggingface-cli login`

然后，我们只需运行 `mlagents-push-to-hf`。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mlpush.png" alt="ml learn function" width="100%">

我们定义了 4 个参数：

1. `--run-id`: 训练运行的名称 ID。
2. `--local-dir`: 智能体保存的位置，它是 `results/<run_id name>`，在我的例子中是 `results/Huggy2`。
3. `--repo-id`: 你想创建或更新的 Hugging Face 仓库名称。它始终是 `<你的 huggingface 用户名>/<仓库名称>`。如果仓库不存在，**它将自动创建**。
4. `--commit-message`: 由于 HF 仓库是 git 仓库，你需要定义一条提交消息。

In [ ]:
!mlagents-push-to-hf --run-id="HuggyTraining" --local-dir="./results/Huggy2" --repo-id="ThomasSimonini/ppo-Huggy" --commit-message="Huggy"

如果一切顺利，在流程结束时你应该看到如下内容（但 URL 会有所不同 😆）：

```
Your model is pushed to the hub. You can view your model here: https://huggingface.co/ThomasSimonini/ppo-Huggy
```

这是指向你模型仓库的链接。该仓库包含一个解释如何使用模型的模型卡片、你的 Tensorboard 日志和你的配置文件。**最棒的是它是一个 git 仓库，这意味着你可以有不同的提交，通过新的推送更新你的仓库，开启 Pull Requests 等等。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/modelcard.png" alt="ml learn function" width="100%">

但现在最精彩的部分来了：**能够在线与 Huggy 玩耍 👀。**

## 与你的 Huggy 🐕 玩耍

这一步最简单：

- 在浏览器中打开 Huggy 游戏：https://huggingface.co/spaces/ThomasSimonini/Huggy

- 点击 **Play with my Huggy model**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/load-huggy.jpg" alt="load-huggy" width="100%">

1. 在第一步中，输入你的用户名（用户名区分大小写：例如，我的用户名是 ThomasSimonini 而不是 thomassimonini 或 ThOmasImoNInI），然后点击搜索按钮。

2. 在第二步中，选择你的模型仓库。

3. 在第三步中，**选择你想回放哪个模型**：
  - 我有多个模型，因为我们每 500,000 个时间步保存了一个模型。
  - 但由于我想要最近的那个，我选择 `Huggy.onnx`。

👉 尝试使用不同步数的模型来观察智能体的进步是非常有趣的事情。

祝贺你完成这个加餐单元！

你现在可以坐下来享受与你的 Huggy 🐶 玩耍的乐趣了。别忘了通过与你的朋友分享 Huggy 来**传递爱心 🤗**。如果你在社交媒体上分享，**请通过 @huggingface 标记我们，并通过 @simoninithomas 标记我**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-cover.jpeg" alt="Huggy cover" width="100%">


## 继续学习，保持出色 🤗